# 🧠 Rock-Paper-Scissors — YOLO Classification

---

## 🎯 Objective

Train a **YOLO classification model** (YOLOv26n-cls) to classify hand gesture images
into Rock, Paper, or Scissors.

### What You Will Learn

| Topic | Description |
|-------|-------------|
| YOLO framework | How Ultralytics YOLO handles data loading, augmentation, and training |
| Transfer learning | Fine-tune a pre-trained YOLO classification backbone |
| Built-in augmentation | YOLO applies `randaugment`, flips, HSV shifts, and more automatically |
| Training curves | Visualize loss and accuracy from YOLO's CSV results |
| Inference | Predict on new images with the trained model |

### Prerequisites

- Run **`data_split.ipynb`** first to generate `split_data/`.

> 💡 **Note:** Unlike Keras notebooks, YOLO manages its own data pipeline, augmentation,
> and training loop. We interact with it through a high-level API.

---

## Step 1 — Import Libraries

In [ ]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
import random
from PIL import Image

print('✅ Ultralytics YOLO imported successfully.')

---

## Step 2 — Configuration

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║        CONFIGURATION — CHANGE HERE           ║
# ╚══════════════════════════════════════════════╝

DATA_DIR      = '../split_data'
MODEL_NAME    = 'yolo11n-cls.pt'    # Pre-trained YOLO classification model
IMAGE_SIZE    = 150
BATCH_SIZE    = 32
EPOCHS        = 20
PATIENCE      = 5
WEIGHTS_DIR   = '../weights/yolo'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

---

## Step 3 — Visualize Data Samples

Before training, let's look at some images from the training set.

In [ ]:
def show_batch_from_dir(data_dir, split='train', title='Sample Images', n=9):
    """Display a grid of sample images from a directory-based dataset."""
    split_path = Path(data_dir) / split
    classes = sorted([d.name for d in split_path.iterdir() if d.is_dir() and not d.name.startswith('.')])

    plt.figure(figsize=(8, 8))
    for i in range(min(n, 9)):
        cls = classes[i % len(classes)]
        cls_path = split_path / cls
        images = [f for f in cls_path.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp')]
        if images:
            img_path = random.choice(images)
            img = Image.open(img_path)
            ax = plt.subplot(3, 3, i + 1)
            plt.imshow(img)
            plt.title(cls)
            plt.axis('off')
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

show_batch_from_dir(DATA_DIR, 'train', 'Training Samples')
show_batch_from_dir(DATA_DIR, 'test',  'Test Samples')

---

## Step 4 — Load the Pre-trained Model

We start with a YOLO model pre-trained on ImageNet.
The classification head will be replaced to match our 3 classes.

In [ ]:
model = YOLO(MODEL_NAME)
print(f'✅ Loaded model: {MODEL_NAME}')

---

## Step 5 — Train the Model

YOLO's `.train()` handles everything:

| Feature | Handled by YOLO |
|---------|-----------------|
| Data loading | Reads from `train/`, `val/`, `test/` folders automatically |
| Augmentation | `randaugment`, horizontal flip, HSV shifts, erasing, etc. |
| Optimizer | Auto-selects AdamW with tuned LR and momentum |
| Early stopping | Built-in `patience` parameter |
| Checkpointing | Saves `best.pt` and `last.pt` automatically |

> 📌 **YOLO's built-in augmentations** include colour jitter (HSV), flipping, scaling,
> translation, random erasing, and `randaugment`. No manual augmentation layers needed.

In [ ]:
results = model.train(
    data=DATA_DIR,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
)

print(f'\n✅ Training complete!')

---

## Step 6 — Training Curves

YOLO saves detailed training metrics to a CSV file. Let's plot them.

In [ ]:
# Load training results
df = pd.read_csv(f'{results.save_dir}/results.csv')
df.columns = df.columns.str.strip()

# Print summary table
print(df[['epoch', 'train/loss', 'metrics/accuracy_top1', 'metrics/accuracy_top5']].to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df['epoch'], df['train/loss'], marker='o', label='Train Loss')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['epoch'], df['metrics/accuracy_top1'], marker='o', label='Top-1 Accuracy')
axes[1].plot(df['epoch'], df['metrics/accuracy_top5'], marker='o', label='Top-5 Accuracy', linestyle='--')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('YOLO — Training Curves', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---

## Step 7 — Save Best Model to Weights Directory

YOLO saves models in its own `runs/` directory. Let's copy the best one to our organized `weights/` folder.

In [ ]:
import shutil

best_model_src = Path(results.save_dir) / 'weights' / 'best.pt'
best_model_dst = Path(WEIGHTS_DIR) / 'best.pt'

if best_model_src.exists():
    shutil.copy2(best_model_src, best_model_dst)
    print(f'✅ Best model copied to: {best_model_dst}')
else:
    print('⚠️  best.pt not found — check the training output.')

---

## 📚 Summary

| Metric | Value |
|--------|-------|
| Architecture | YOLOv26n-cls (pre-trained on ImageNet) |
| Augmentation | YOLO built-in (randaugment, HSV, flip, erasing) |
| Early stopping | Built-in patience parameter |
| Model saved to | `weights/yolo/best.pt` |

### Key Takeaways for Students

1. **YOLO is a high-level framework** — it handles data loading, augmentation, optimization, and checkpointing for you.
2. **Pre-trained models converge faster** — the ImageNet backbone already knows visual features.
3. **Always inspect training curves** — they tell you if the model is learning properly.
4. **YOLO is great for rapid prototyping** — minimal code, strong results.